# 🔁 Notebook: Apache Kafka — Delivery Guarantees, Consumer Groups & Retention

The first notebook got a broker running and moved single messages back and forth. This notebook is about the features that separate "I can call `.produce()`" from actually understanding how Kafka behaves under real conditions: what happens when a message might get lost or duplicated, how multiple consumers split up work and recover from failure, how long data actually stays around, and what to do when things go wrong.

These are also, not coincidentally, the topics that dominate Kafka interviews — delivery guarantees and consumer-group rebalancing in particular come up constantly, because they're where subtle bugs live in real systems.

> This notebook assumes the broker and Kafka UI from [Notebook 1](./02_1_intro_kafka.ipynb) are still running. If not, re-run its Docker setup cells first (section 4) before continuing here.

## 📚 Sources

- [Kafka: Semantics](https://kafka.apache.org/documentation/#semantics) (delivery guarantees)
- [Kafka: Idempotent Producer](https://kafka.apache.org/documentation/#semantics)
- [Kafka: Consumer Groups](https://kafka.apache.org/documentation/#intro_consumers)
- [Kafka: Log Compaction](https://kafka.apache.org/documentation/#compaction)
- [Kafka: Topic Configs](https://kafka.apache.org/documentation/#topicconfigs)
- [confluent-kafka-python: Consumer API](https://docs.confluent.io/kafka-clients/python/current/overview.html#kafka-client-configuration)


In [1]:
# Re-establish the connection details and a couple of helper objects we'll
# reuse throughout this notebook.
from confluent_kafka import Producer, Consumer, KafkaException
from confluent_kafka.admin import AdminClient, NewTopic
import json, time

BOOTSTRAP = "localhost:9092"
admin = AdminClient({"bootstrap.servers": BOOTSTRAP})
print("connected brokers:", admin.list_topics(timeout=5).brokers)

connected brokers: {1: BrokerMetadata(1, localhost:9092)}


## 1. Delivery guarantees

A delivery guarantee describes what can happen to a message when a producer, broker, or consumer fails. Kafka supports three options, chosen through configuration:

- **At-most-once** — a message can be lost, but is never processed twice. This happens when a consumer marks the message as handled *before* processing it and then crashes.
- **At-least-once** — a message is never lost, but can be processed twice. Process the message first, then mark it as handled. If the consumer crashes in between, the next consumer processes it again.
- **Exactly-once (EOS)** — each message has an effect exactly once, even after failures. Kafka combines an **idempotent producer** (repeating a send has the same effect as sending it once; retries are deduplicated) with **transactions** that record produced messages and processed offsets together.

> **Important:** Exactly-once applies only inside Kafka. An external side effect, such as charging a payment or writing to another database, can still happen twice. Make consumer logic idempotent (safe to run twice) whenever possible.

### The idempotent producer

Without idempotence, a dropped acknowledgment forces the producer to guess: resend (risking a duplicate if the original write actually succeeded) or give up (risking data loss). `enable.idempotence=true` fixes this by having the producer tag every message with a sequence number; the broker recognizes and silently drops a retried duplicate. It's a single config flag with no application code changes required, and is the modern default recommendation for any producer that cares about correctness — as of recent Kafka client versions it's on by default.


In [2]:
# An explicitly idempotent, "safe" producer configuration:
safe_producer = Producer({
    "bootstrap.servers": BOOTSTRAP,
    "enable.idempotence": True,   # dedupe retried sends on the broker
    "acks": "all",                # wait for every in-sync replica to acknowledge (see below)
    "retries": 5,
})

topic = "orders-placed"
admin.create_topics([NewTopic(topic, num_partitions=3, replication_factor=1)])
time.sleep(1)

def report(err, msg):
    if err:
        print("FAILED:", err)
    else:
        print(f"acked: partition={msg.partition()} offset={msg.offset()}")

safe_producer.produce(topic, key="order-1", value=json.dumps({"order_id": 1, "total": 42.50}), callback=report)
safe_producer.flush()

%4|1789473879.845|GETPID|rdkafka#producer-2| [thrd:main]: Failed to acquire idempotence PID from broker localhost:9092/1: Broker: Coordinator load in progress: retrying


acked: partition=1 offset=0


0

### `acks`: how many replicas must confirm a write

`acks` controls how many brokers must confirm a write before the producer considers it successful — a direct trade-off between durability and latency/throughput:

- `acks=0` — don't wait for any acknowledgment at all. Fastest, but a message can be silently lost.
- `acks=1` (default in many client configs) — wait for the partition's **leader** broker only. Safe against most failures, but if the leader crashes right after acknowledging and before followers replicated the write, that message is lost.
- `acks=all` (equivalent to `acks=-1`) — wait for every **in-sync replica** to confirm. The strongest guarantee, at the cost of latency. Combined with the broker-side setting `min.insync.replicas` (how many replicas must be in-sync for a write to even be accepted), this is the standard configuration for data you can't afford to lose. More on replication in [Notebook 3](./02_3_kafka_ecosystem_and_scaling.ipynb).


## 2. Consumer groups and rebalancing

A **consumer group** is a set of consumers that share the work of reading a topic: Kafka assigns each partition to exactly one consumer *within* the group at any given time. Add more consumers (up to the number of partitions) and you get more parallelism; add a consumer beyond the partition count and it sits idle, since a partition can't be split further between two consumers in the same group.

A **rebalance** is the process of reassigning partitions among a group's consumers, triggered whenever group membership changes: a consumer joins, a consumer leaves (cleanly, via `.close()`, or by crashing/timing out), or the topic's partition count changes. During a rebalance, consumption briefly pauses while partitions are handed out to their new owners.

Let's see this directly: start two consumers in the same group, watch how a 3-partition topic is split 2/1 between them, then simulate one leaving.


In [3]:
# Polling triggers the join/rebalance protocol - assignment only settles
# after the group coordinator has heard from every member, which can take a
# couple of poll() calls, so we poll in a short loop rather than just once.
def wait_for_assignment(consumer, attempts=10, timeout=1.0):
    for _ in range(attempts):
        consumer.poll(timeout=timeout)
        if consumer.assignment():
            return
    print("warning: no partitions assigned yet - the group may still be rebalancing")

def wait_for_rebalance(consumer, attempts=10, timeout=1.0):
    """Like wait_for_assignment, but keeps polling for the full attempt budget
    even once *some* assignment exists - useful right after another member
    left the group, since the survivor's assignment changes from a non-empty
    starting point and a single 'is it non-empty yet' check would return too
    early."""
    # waits a maximum of timeout * attempts seconds for the consumer to get a new assignment
    for _ in range(attempts):
        consumer.poll(timeout=timeout)

# Two consumers, same group.id, subscribing to our 3-partition orders-placed topic.
group_id = "order-processors"

consumer_a = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": group_id, "auto.offset.reset": "earliest"})
consumer_b = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": group_id, "auto.offset.reset": "earliest"})

consumer_a.subscribe([topic])
consumer_b.subscribe([topic])

wait_for_assignment(consumer_a)
wait_for_assignment(consumer_b)

print("consumer_a got partitions:", [p.partition for p in consumer_a.assignment()])
print("consumer_b got partitions:", [p.partition for p in consumer_b.assignment()])

consumer_a got partitions: [0, 1]
consumer_b got partitions: [2]


In [4]:
# Now let consumer_b leave the group cleanly. Kafka immediately triggers a
# rebalance and hands its partition(s) over to consumer_a.
consumer_b.close()

# consumer_a needs to keep polling to notice the rebalance and pick up the
# newly assigned partition(s) - it already owned some partitions before this,
# so we wait out the full budget rather than stopping at the first (already
# non-empty) assignment we see.
wait_for_rebalance(consumer_a)
print("consumer_a now owns:", [p.partition for p in consumer_a.assignment()])

consumer_a.close()

consumer_a now owns: [0, 1, 2]


Switch to Kafka UI's **Consumers** page while re-running the two cells above (undoing `.close()` calls by re-creating the consumers) to watch the group's state and partition ownership change live — this is exactly the kind of behavior that's much easier to build intuition for visually than from printed partition numbers alone.

> **Consumer lag**, one of the most important operational metrics in any Kafka system, is simply *the difference between the latest offset written to a partition and the offset a consumer group has last committed for it* — i.e., how far behind the consumer is. Kafka UI shows this per-partition and per-group; in production it's usually the #1 metric you'd alert on, since a growing lag means consumers can't keep up with producers.


## 3. Offset management: auto-commit vs. manual commit

By default (`enable.auto.commit=True`), the client periodically commits the offset of the last message returned by `.poll()`, in the background, on a timer — simple, but it means a crash between "message returned by poll()" and "actually finished processing it" can lose progress silently (the *next* consumer to take over that partition will skip past it, thinking it was already handled).

**Manual commits** (`enable.auto.commit=False`, then calling `.commit()` yourself after successfully processing a message) give you control over exactly when "done" is recorded — the basis of the at-least-once pattern from section 1: process first, commit after.


In [5]:
manual_consumer = Consumer({
    "bootstrap.servers": BOOTSTRAP,
    "group.id": "order-processors-manual",
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False,
})
manual_consumer.subscribe([topic])

processed = 0
while processed < 1:
    msg = manual_consumer.poll(timeout=3.0)
    if msg is None or msg.error():
        continue

    # "Process" the message - in a real app this might write to a database,
    # call an API, etc. Only commit once that has actually succeeded.
    order = json.loads(msg.value())
    print("processing order:", order)

    print("committing message:", msg)

    manual_consumer.commit(msg)  # commit *after* successful processing
    print(msg)
    processed += 1

manual_consumer.close()

processing order: {'order_id': 1, 'total': 42.5}
committing message: <confluent_kafka.cimpl.Message object at 0x108861870>


**Replaying messages** — re-processing data a consumer group has already committed past, e.g. after fixing a bug in how events were handled — is just a matter of moving its committed offset backwards (or resetting it to the beginning) before consuming again. This is one of Kafka's most valuable properties compared to a traditional queue, where a consumed message is typically gone: as long as a topic's retention hasn't expired the data yet (see next section), *any* consumer group can rewind and re-read history.


In [ ]:
# Reset the "order-processors-manual" group's offsets back to the beginning
# of every partition it's assigned to, then confirm the first message it
# reads next is the very first one again.
from confluent_kafka import TopicPartition

replay_consumer = Consumer({
    "bootstrap.servers": BOOTSTRAP,
    "group.id": "order-processors-manual",
    "enable.auto.commit": False,
})
replay_consumer.subscribe([topic])
wait_for_assignment(replay_consumer)  # defined in section 2 above

assigned = replay_consumer.assignment()
print("replay consumer assigned partitions:", [p.partition for p in assigned])
for tp in assigned:
    # .seek moves the consumer's "cursor" to the given offset for the given partition, so the next poll() will return that message (or None if there is no such offset). Offset 0 is always the first message in a partition.
    replay_consumer.seek(TopicPartition(tp.topic, tp.partition, 0))  # offset 0 = beginning

for _ in range(5):
    msg = replay_consumer.poll(timeout=2.0)
    if msg and not msg.error():
        print("re-read from the beginning:", json.loads(msg.value()))
        break

replay_consumer.close()

replay consumer assigned partitions: [0, 1, 2]
re-read from the beginning: {'order_id': 1, 'total': 42.5}


## 4. Retention and log compaction

Kafka doesn't keep data forever by default, but "forever" is also a valid choice — retention is fully configurable per topic. Two independent mechanisms control this, and picking the right one is a common design question:

- **Time/size-based retention** (`retention.ms`, `retention.bytes`) — the default. Kafka deletes whole log segments once they're older than the configured time (default 7 days) or the partition exceeds the configured size, regardless of whether every message has been consumed. This is the right choice for genuine *event streams* — an audit log, a stream of clicks — where you care about history up to a point, not a "current value".
- **Log compaction** (`cleanup.policy=compact`) — instead of deleting by age, Kafka keeps only the **most recent value for each key**, discarding older records with the same key (asynchronously, via a background "cleaner" thread — not instantly). This turns a topic into something closer to a changelog of a key-value table: "what is the latest known state of *each* key", forever. This is exactly the mechanism Kafka Streams uses internally for state stores, and the same idea a CDC (change-data-capture) pipeline uses to mirror a database table into Kafka.

> **Interview framing:** "when would you use compaction over normal retention?" — whenever a topic represents *current state per key* rather than a *sequence of facts*. A user-profile-updates topic (keyed by `user_id`, only the latest profile matters) → compaction. A page-view-events topic (every view matters, none of them are "replaced" by a later one) → time-based retention.


In [7]:
# Build a compacted topic: "latest known status per order_id".
compacted_topic = "order-status-latest"
admin.create_topics([NewTopic(
    compacted_topic,
    num_partitions=1,
    replication_factor=1,
    config={"cleanup.policy": "compact", "segment.ms": "100", "min.cleanable.dirty.ratio": "0.01"},
)])
time.sleep(1)

# Produce several updates for the SAME key (order_id=1) - each new value
# represents "order 1 is now in this state".
for status in ["created", "paid", "shipped", "delivered"]:
    safe_producer.produce(compacted_topic, key="order-1", value=json.dumps({"status": status}))
safe_producer.produce(compacted_topic, key="order-2", value=json.dumps({"status": "created"}))
safe_producer.flush()

print("produced 4 updates for order-1 and 1 for order-2.")
print("Compaction runs asynchronously in the background, so reading immediately after")
print("producing will likely still show all 4 - this is expected, not a bug.")

produced 4 updates for order-1 and 1 for order-2.
Compaction runs asynchronously in the background, so reading immediately after
producing will likely still show all 4 - this is expected, not a bug.


Compaction doesn't run instantly (it's a background process, and we set aggressive `segment.ms`/`min.cleanable.dirty.ratio` values above purely so it has a chance to run within this notebook rather than its multi-hour production defaults) — in Kafka UI, revisit the `order-status-latest` topic a minute or two from now and you should eventually see only the latest message for `order-1` (`"delivered"`) survive, while `order-2`'s single message is untouched (nothing to compact away yet).


## 5. Serialization: JSON now, schemas later

Every example so far used `json.dumps`/`json.loads` — readable, ubiquitous, and a perfectly good starting point. It has two costs that matter as a system grows: JSON is verbose on the wire (more bytes than a binary format, for high-throughput topics that adds up), and it enforces **nothing** about structure — a producer can send a field with the wrong type or a typo'd key and no one notices until a consumer crashes trying to read it.

Binary formats like **Avro** and **Protobuf**, combined with a **Schema Registry** (a separate service that stores and versions each topic's schema), solve both: smaller messages, and every producer/consumer validates against a shared, versioned contract. The registry additionally enforces **compatibility rules** (e.g. "a new schema version must still be readable by consumers using the previous version") when a schema changes, which is what actually prevents a producer's innocent field rename from breaking every downstream consumer at 3am.

We won't stand up a full Schema Registry in this notebook — it's a separate service with its own container and its own learning curve — but knowing *why* it exists and *what problem it solves* is exactly the kind of thing that comes up in interviews about running Kafka in production. JSON remains a completely legitimate choice for lower-throughput topics or when consumers are less tightly coupled to a fixed schema.


## 6. Handling failures: retries and dead-letter topics

Two independent failure modes need separate handling:

- **Producer-side failures** — a broker is temporarily unreachable, a leader election is in progress, etc. The client's `retries` config (and `retry.backoff.ms`) handles this automatically for transient errors; combined with `enable.idempotence=True` (section 1), retried sends can't create duplicates.
- **Consumer-side failures** — the message itself can't be processed (e.g. it fails validation, or a downstream API call it triggers keeps failing). Retrying forever would block that partition indefinitely for every message behind it. The standard pattern is a **dead-letter topic (DLT)**: after N failed attempts, the consumer publishes the poison message (plus the error) to a separate `*-dead-letter` topic instead of retrying forever, then commits its offset and moves on — unblocking the partition. The dead-letter topic is then inspected/reprocessed manually or by a separate recovery process.


In [8]:
# A minimal dead-letter pattern: retry a "processing" step a few times,
# and if it keeps failing, route the message to a dead-letter topic instead
# of getting stuck.
dead_letter_topic = "orders-placed-dead-letter"
admin.create_topics([NewTopic(dead_letter_topic, num_partitions=1, replication_factor=1)])
time.sleep(1)

def process_order(order):
    """Pretend this sometimes fails - e.g. a downstream payment API call."""
    if order.get("total", 0) < 0:
        raise ValueError("negative order total")
    return True

def handle_message(msg, max_retries=3):
    order = json.loads(msg.value())
    for attempt in range(1, max_retries + 1):
        try:
            process_order(order)
            return True
        except Exception as e:
            print(f"attempt {attempt} failed: {e}")
    # Exhausted retries - route to the dead-letter topic instead of blocking.
    safe_producer.produce(
        dead_letter_topic,
        key=msg.key(),
        value=json.dumps({"original": order, "error": "exceeded max retries"}),
    )
    safe_producer.flush()
    print("routed to dead-letter topic:", dead_letter_topic)
    return False

# Demonstrate with a deliberately invalid order.
safe_producer.produce(topic, key="order-bad", value=json.dumps({"order_id": 99, "total": -5}))
safe_producer.flush()

dlt_consumer = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": "dlt-demo", "auto.offset.reset": "earliest"})
dlt_consumer.subscribe([topic])
msg = None
for _ in range(10):
    msg = dlt_consumer.poll(timeout=2.0)
    if msg and not msg.error() and json.loads(msg.value()).get("order_id") == 99:
        break
if msg:
    handle_message(msg)
dlt_consumer.close()

attempt 1 failed: negative order total
attempt 2 failed: negative order total
attempt 3 failed: negative order total
routed to dead-letter topic: orders-placed-dead-letter


## 7. Exercises

### Exercise 1: `acks=0` vs. `acks=all`

**Task:** Create two producers against `orders-placed` — one with `acks=0`, one with `acks=all`. Produce 5 messages with each and time how long `producer.flush()` takes for both (use `time.perf_counter()`). Which is faster, and why?


In [9]:
# Your solution here:


<details>
<summary><b>Show Solution</b></summary>

```python
import time

fast_producer = Producer({"bootstrap.servers": BOOTSTRAP, "acks": 0})
strong_producer = Producer({"bootstrap.servers": BOOTSTRAP, "acks": "all"})

start = time.perf_counter()
for i in range(5):
    fast_producer.produce(topic, value=json.dumps({"i": i}))
fast_producer.flush()
print("acks=0 took:", time.perf_counter() - start, "seconds")

start = time.perf_counter()
for i in range(5):
    strong_producer.produce(topic, value=json.dumps({"i": i}))
strong_producer.flush()
print("acks=all took:", time.perf_counter() - start, "seconds")
```

`acks=0` is faster because the producer doesn't wait for any broker confirmation before considering the send complete - it also means a message could be silently lost if the broker never actually received it. `acks=all` waits for every in-sync replica to confirm, which is slower but only reports success once the write is durably safe.

</details>


### Exercise 2: Three consumers, three partitions, one crash

**Task:** `orders-placed` has 3 partitions. Start three consumers in a new group `order-processors-3`, wait for each one to receive its assignment (reuse `wait_for_assignment` from section 2), and print each one's assigned partitions (should be 1 each). Then `.close()` one of them and show, after waiting for the remaining two to rebalance (`wait_for_rebalance`), how the now-orphaned partition gets picked up by one of the survivors.


In [10]:
# Your solution here:


<details>
<summary><b>Show Solution</b></summary>

```python
group_id = "order-processors-3"
consumers = [
    Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": group_id, "auto.offset.reset": "earliest"})
    for _ in range(3)
]
for c in consumers:
    c.subscribe([topic])
for c in consumers:
    wait_for_assignment(c)

for i, c in enumerate(consumers):
    print(f"consumer {i} owns:", [p.partition for p in c.assignment()])

consumers[0].close()

for c in consumers[1:]:
    wait_for_rebalance(c)
for i, c in enumerate(consumers[1:], start=1):
    print(f"consumer {i} now owns:", [p.partition for p in c.assignment()])

for c in consumers[1:]:
    c.close()
```

</details>


### Exercise 3: A compacted "latest score per player" topic

**Task:** Create a compacted topic `leaderboard` (`cleanup.policy=compact`). Produce three score updates each for `player-1` and `player-2` (six messages total, increasing scores). Explain in a comment why, once compaction has run, `.get()`-style full reads of this topic would show exactly two messages, not six.


In [11]:
# Your solution here:


<details>
<summary><b>Show Solution</b></summary>

```python
admin.create_topics([NewTopic(
    "leaderboard",
    num_partitions=1,
    replication_factor=1,
    config={"cleanup.policy": "compact", "segment.ms": "100", "min.cleanable.dirty.ratio": "0.01"},
)])
time.sleep(1)

for player in ["player-1", "player-2"]:
    for score in [10, 25, 40]:
        safe_producer.produce("leaderboard", key=player, value=json.dumps({"score": score}))
safe_producer.flush()

# Once the background compaction (log cleaner) thread has processed this
# partition's segments, only the LAST message written for each key survives
# - here, {"score": 40} for both player-1 and player-2 - because compaction
# deletes every older record sharing a key, keeping just the newest value.
# Reading the full topic afterwards therefore shows 2 messages, not 6.
```

</details>


## Cleanup

If you're moving straight on to [Notebook 3](./02_3_kafka_ecosystem_and_scaling.ipynb), you can leave the containers running. Otherwise, tear everything down:


In [12]:
!docker rm -f kafka-broker kafka-ui
!docker network rm kafka-net

%6|1789393942.357|FAIL|rdkafka#producer-2| [thrd:localhost:9092/1]: localhost:9092/1: Disconnected: connection closed by peer: receive 0 after POLLIN (after 34621ms in state UP)
%6|1789393942.357|FAIL|rdkafka#producer-1| [thrd:localhost:9092/1]: localhost:9092/1: Disconnected: connection closed by peer: receive 0 after POLLIN (after 35142ms in state UP)


kafka-broker
kafka-ui


kafka-net
